# Financial model comparison with lm-evaluation-harness

Evaluate the saved Qwen, SmolLM2, and Pythia adapters, each against its original
base model, sequentially on one Colab NVIDIA GPU (A100, L4, or T4). No retraining
is performed. Use a **fresh evaluation runtime**, separate from training.

This notebook is self-contained: evaluation utilities are embedded below, so no
repository clone or additional Python upload is required. Only your completed
training run folders on Drive are needed.

The default pilot scores 20 unseen passages and 20 literal dollar-amount
extraction questions. This narrowly tests financial prose prediction and copying
an explicitly stated amount. It does not measure general financial reasoning.
Increase the frozen suite to 200 passages / 100 questions for a larger extraction
study; reviewed custom questions can be added before freezing a new suite.

Exact normalized training/validation document hashes from all three runs are
excluded. Related passages and pretraining contamination cannot be ruled out.
Your scratch model is not included: it needs separate harness integration and
document-level training provenance.


## 1. Install a separate evaluation environment

Keep Colab's CUDA-enabled PyTorch. Restart after installation if any listed library was already imported. This notebook pins harness 0.4.9 and records the complete installed package set.

In [ ]:
import subprocess, sys
PACKAGES = [
    "lm_eval==0.4.9", "transformers==4.51.3", "peft==0.15.2",
    "accelerate==1.6.0", "datasets==3.5.0", "huggingface_hub==0.30.2",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *PACKAGES])
print("Installation complete. Restart if these libraries were already imported.")


## 2. Embedded evaluation utilities

This cell defines document hashing, strict answer scoring, custom harness tasks, and the model evaluation loop. It does not download or train models.

In [ ]:
import os
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import hashlib
import csv
import json
import math
import re
import unicodedata
import gc
import time
from decimal import Decimal
from pathlib import Path


MONEY = re.compile(r"\$\s*\d[\d,]*(?:\.\d+)?(?:\s+(?:thousand|million|billion|trillion))?", re.I)


def normalize_text(text):
    return unicodedata.normalize("NFKC", text).replace("\x00", "").strip()


def normalized_hash(text):
    return hashlib.sha256(" ".join(normalize_text(text).split()).encode("utf-8")).hexdigest()


def suite_digest(suite):
    payload = json.dumps(suite, sort_keys=True, ensure_ascii=False, allow_nan=False)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def canonical_answer(value):
    value = " ".join(value.strip().strip('"').casefold().split())
    match = re.fullmatch(
        r"(\$)?\s*([+-]?\d[\d,]*(?:\.\d+)?)\s*(thousand|million|billion|trillion)?\s*(%)?",
        value,
    )
    if match:
        currency, number, scale, percent = match.groups()
        multiplier = {None: 1, "thousand": 10**3, "million": 10**6,
                      "billion": 10**9, "trillion": 10**12}[scale]
        number = Decimal(number.replace(",", "")) * multiplier
        return (currency or "") + format(number.normalize(), "f") + (percent or "")
    return value


def score_answer(response, answers):
    return float(canonical_answer(response) in {canonical_answer(a) for a in answers})


def load_runs(run_dirs):
    runs, excluded = [], set()
    for label, directory in run_dirs.items():
        root = Path(directory)
        adapter = root / "adapter"
        required = [root / "run_config.json", root / "split_manifest.json",
                    adapter / "adapter_config.json", adapter / "adapter_model.safetensors",
                    adapter / "tokenizer_config.json"]
        missing = [str(p) for p in required if not p.is_file()]
        if missing:
            raise ValueError(f"Incomplete run {label}: {missing}")
        config = json.loads(required[0].read_text())
        if not re.fullmatch(r"[0-9a-f]{40}", config.get("model_sha", "")):
            raise ValueError(f"Missing immutable model revision: {label}")
        adapter_config = json.loads((adapter / "adapter_config.json").read_text())
        if adapter_config.get("base_model_name_or_path") != config["MODEL_ID"]:
            raise ValueError(f"Adapter/base model mismatch: {label}")
        if config.get("MAX_DOC_CHARS") != 32768:
            raise ValueError("Overlap checking currently requires MAX_DOC_CHARS=32768 in every run.")
        manifest = json.loads(required[1].read_text())
        if set(manifest) != {"train", "eval"} or not manifest["train"]:
            raise ValueError(f"Invalid split manifest: {label}")
        for split in manifest.values():
            for row in split:
                digest = row.get("sha256", "")
                if not re.fullmatch(r"[0-9a-f]{64}", digest):
                    raise ValueError(f"Invalid document hash: {label}")
                excluded.add(digest)
        runs.append(dict(label=label, model_id=config["MODEL_ID"],
                         model_sha=config["model_sha"], dataset_id=config["DATASET_ID"],
                         dataset_sha=config["dataset_sha"], adapter=str(adapter),
                         training_config=config,
                         manifest_sha256=hashlib.sha256(required[1].read_bytes()).hexdigest(),
                         adapter_sha256=hashlib.sha256(
                             (adapter / "adapter_model.safetensors").read_bytes()).hexdigest()))
    if not runs:
        raise ValueError("At least one completed run is required.")
    if len({(r["dataset_id"], r["dataset_sha"]) for r in runs}) != 1:
        raise ValueError("Training runs use different dataset revisions; align their source data first.")
    return runs, excluded


def build_suite(rows, excluded, dataset_sha, document_count=200, question_count=20):
    if not 1 <= question_count <= document_count:
        raise ValueError("Require 1 <= question_count <= document_count.")
    documents, questions, seen = [], [], set()
    for row in rows:
        if not isinstance(row.get("source"), str):
            continue
        full = normalize_text(row["source"])
        capped = full[:32768].strip()
        digest = normalized_hash(capped)
        if digest in excluded or digest in seen or len(capped) < 80:
            continue
        seen.add(digest)
        passage = capped[:1200].strip()
        match = MONEY.search(passage)
        if not match and document_count - len(documents) <= question_count - len(questions):
            continue
        document_id = f"financial_{len(documents):04d}"
        document = dict(id=document_id, text=passage, document_sha256=digest,
                        capped_document=capped, passage_sha256=normalized_hash(passage),
                        source_label=str(row.get("dataset", "unknown")),
                        source_file=str(row.get("source_file", "")), source_index=str(row.get("index", "")))
        if len(documents) < document_count:
            documents.append(document)
            if match and len(questions) < question_count:
                questions.append(dict(
                    id=f"amount_{len(questions):04d}", document_id=document_id,
                    context=passage,
                    question="What is the first dollar-denominated amount in the passage? Copy the amount including $ and any scale word. Return only the amount.",
                    answers=[match.group(0)], category="literal_amount_extraction",
                ))
        if len(documents) == document_count and len(questions) == question_count:
            break
    if len(documents) != document_count or len(questions) != question_count:
        raise ValueError("Insufficient unseen passages with dollar amounts. Increase scan budget or choose a new suite.")
    suite = dict(schema_version=1, dataset_sha=dataset_sha, documents=documents,
                 questions=questions, purpose="pilot", reference_policy="literal_first_dollar_amount")
    audit_suite(suite, excluded)
    return suite


def audit_suite(suite, excluded):
    if suite.get("schema_version") != 1 or not suite.get("documents") or not suite.get("questions"):
        raise ValueError("Invalid or empty evaluation suite.")
    ids, hashes = set(), set()
    documents = {}
    for doc in suite["documents"]:
        if doc["id"] in ids or doc["document_sha256"] in hashes:
            raise ValueError("Duplicate evaluation document.")
        ids.add(doc["id"])
        hashes.add(doc["document_sha256"])
        if normalized_hash(doc["capped_document"]) != doc["document_sha256"]:
            raise ValueError("Document content/hash mismatch.")
        if doc["document_sha256"] in excluded:
            raise ValueError("Test document overlaps a recorded training/validation document.")
        if not doc["text"] or doc["text"] not in doc["capped_document"]:
            raise ValueError("Passage is not grounded in its source document.")
        if normalized_hash(doc["text"]) != doc["passage_sha256"]:
            raise ValueError("Passage content/hash mismatch.")
        documents[doc["id"]] = doc
    question_ids = set()
    for question in suite["questions"]:
        doc = documents.get(question["document_id"])
        if question["id"] in question_ids or doc is None or question["context"] != doc["text"]:
            raise ValueError("Invalid question ID, source, or context.")
        question_ids.add(question["id"])
        if not question["answers"] or not all(isinstance(a, str) and a.strip() for a in question["answers"]):
            raise ValueError("Reference answers are required.")
        if question.get("category") == "literal_amount_extraction":
            match = MONEY.search(question["context"])
            if not match or question["answers"] != [match.group(0)]:
                raise ValueError("Extraction reference does not match the passage.")
        elif not question.get("reviewed", False):
            raise ValueError("Custom calculation/comprehension questions require reviewed=true.")


def qa_prompt(doc):
    return f"Passage:\n{doc['context']}\n\nQuestion: {doc['question']}\nAnswer:"


def qa_results(doc, results):
    return {"acc": score_answer(results[0], doc["answers"])}


def rolling_results(doc, results):
    return {"bits_per_byte": (float(results[0]), len(doc["text"].encode("utf-8")))}


def aggregate_bpb(items):
    return -sum(ll for ll, _ in items) / (sum(count for _, count in items) * math.log(2))


def make_tasks(suite_path):
    from lm_eval.api.task import ConfigurableTask

    suite = json.loads(Path(suite_path).read_text())
    stem = Path(suite_path).parent
    text_file, qa_file = stem / "financial_text.jsonl", stem / "financial_qa.jsonl"
    for path, records in [(text_file, suite["documents"]), (qa_file, suite["questions"])]:
        path.write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in records))
    common = dict(dataset_path="json", test_split="test", num_fewshot=0)
    text = ConfigurableTask(config=dict(
        **common, task="financial_prose", dataset_kwargs={"data_files": {"test": str(text_file)}},
        output_type="loglikelihood_rolling", doc_to_text="", doc_to_target="{{text}}",
        process_results=rolling_results,
        metric_list=[dict(metric="bits_per_byte", aggregation=aggregate_bpb, higher_is_better=False)],
    ))
    qa = ConfigurableTask(config=dict(
        **common, task="financial_qa", dataset_kwargs={"data_files": {"test": str(qa_file)}},
        output_type="generate_until", doc_to_text=qa_prompt, doc_to_target=lambda d: d["answers"][0],
        process_results=qa_results,
        generation_kwargs=dict(do_sample=False, max_gen_toks=48, until=["\n"]),
        metric_list=[dict(metric="acc", aggregation="mean", higher_is_better=True)],
    ))
    return [text, qa]


def evaluate_runs(runs, suite_path, output_dir, excluded, max_length=1024):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel
    from lm_eval import simple_evaluate
    from lm_eval.models.huggingface import HFLM

    if not torch.cuda.is_available():
        raise RuntimeError("Select a Colab NVIDIA GPU runtime.")
    suite = json.loads(Path(suite_path).read_text())
    audit_suite(suite, excluded)
    digest = suite_digest(suite)
    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=False)
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16
    freeze = dict(suite_sha256=digest, dataset_sha=suite["dataset_sha"],
                  purpose=suite["purpose"], runs=runs, max_length=max_length,
                  dtype=str(dtype), quantization="none", gpu=torch.cuda.get_device_name(0),
                  note="Qwen QLoRA adapter evaluated on an unquantized base; differs from its notebook's NF4 baseline.")
    (output / "evaluation_config.json").write_text(json.dumps(freeze, indent=2))
    summaries = []
    for run in runs:
        if run["dataset_sha"] != suite["dataset_sha"]:
            raise ValueError("Evaluation/training dataset revision mismatch.")
        tokenizer = AutoTokenizer.from_pretrained(run["adapter"])
        if tokenizer.pad_token_id is None:
            tokenizer.pad_token = tokenizer.eos_token
        tokenizer.padding_side = "left"
        for question in suite["questions"]:
            length = len(tokenizer(qa_prompt(question), add_special_tokens=False)["input_ids"])
            if length + 48 > max_length:
                raise ValueError(f"Question {question['id']} exceeds the shared context budget for {run['label']}.")
        for mode in ("baseline", "adapted"):
            tasks = make_tasks(suite_path)
            model = wrapper = None
            try:
                model = AutoModelForCausalLM.from_pretrained(
                    run["model_id"], revision=run["model_sha"], torch_dtype=dtype,
                    device_map={"": 0}, attn_implementation="sdpa",
                )
                if max_length > model.config.max_position_embeddings:
                    raise ValueError("Evaluation context exceeds model capacity.")
                if mode == "adapted":
                    model = PeftModel.from_pretrained(model, run["adapter"], is_trainable=False)
                model.eval()
                model.config.use_cache = True
                wrapper = HFLM(pretrained=model, tokenizer=tokenizer, batch_size=1,
                               max_length=max_length, add_bos_token=False)
                torch.cuda.synchronize()
                torch.cuda.reset_peak_memory_stats()
                start = time.perf_counter()
                result = simple_evaluate(
                    model=wrapper, tasks=tasks, num_fewshot=0, log_samples=True,
                    apply_chat_template=False, bootstrap_iters=1000,
                    random_seed=42, numpy_random_seed=42, torch_random_seed=42,
                    fewshot_random_seed=42,
                )
                torch.cuda.synchronize()
                seconds = time.perf_counter() - start
                metrics = result["results"]
                row = dict(model=run["label"], model_id=run["model_id"], mode=mode,
                           suite_sha256=digest,
                           bits_per_byte=metrics["financial_prose"]["bits_per_byte,none"],
                           qa_accuracy=metrics["financial_qa"]["acc,none"],
                           seconds=seconds, peak_allocated_gib=torch.cuda.max_memory_allocated() / 2**30,
                           model_parameters=sum(p.numel() for p in model.parameters()),
                           document_count=len(suite["documents"]), question_count=len(suite["questions"]),
                           dtype=str(dtype))
                if not all(math.isfinite(row[k]) for k in ("bits_per_byte", "qa_accuracy")):
                    raise ValueError("Non-finite evaluation scores.")
                summaries.append(row)
                run_output = output / f"{run['label']}_{mode}"
                run_output.mkdir()
                (run_output / "harness_results.json").write_text(json.dumps(
                    result, indent=2, ensure_ascii=False, default=str))
                with (run_output / "samples.jsonl").open("w") as handle:
                    for task_name, samples in result.get("samples", {}).items():
                        for sample in samples:
                            handle.write(json.dumps(dict(task=task_name, **sample), ensure_ascii=False, default=str) + "\n")
                with (output / "comparison.csv").open("w", newline="") as handle:
                    writer = csv.DictWriter(handle, fieldnames=list(row))
                    writer.writeheader()
                    writer.writerows(summaries)
                (output / "comparison.json").write_text(json.dumps(summaries, indent=2, allow_nan=False))
                print(json.dumps(row, indent=2))
            finally:
                del wrapper, model, tasks
                gc.collect()
                torch.cuda.empty_cache()
    paired = []
    for run in runs:
        baseline, adapted = [r for r in summaries if r["model"] == run["label"]]
        paired.append(dict(model=run["label"],
                           qa_accuracy_change=adapted["qa_accuracy"] - baseline["qa_accuracy"],
                           bits_per_byte_change=adapted["bits_per_byte"] - baseline["bits_per_byte"]))
    (output / "improvement.json").write_text(json.dumps(paired, indent=2, allow_nan=False))
    return summaries


Path("/content/evaluation_utils_snapshot.py").write_text("import hashlib\nimport csv\nimport json\nimport math\nimport re\nimport unicodedata\nimport gc\nimport time\nfrom decimal import Decimal\nfrom pathlib import Path\n\n\nMONEY = re.compile(r\"\\$\\s*\\d[\\d,]*(?:\\.\\d+)?(?:\\s+(?:thousand|million|billion|trillion))?\", re.I)\n\n\ndef normalize_text(text):\n    return unicodedata.normalize(\"NFKC\", text).replace(\"\\x00\", \"\").strip()\n\n\ndef normalized_hash(text):\n    return hashlib.sha256(\" \".join(normalize_text(text).split()).encode(\"utf-8\")).hexdigest()\n\n\ndef suite_digest(suite):\n    payload = json.dumps(suite, sort_keys=True, ensure_ascii=False, allow_nan=False)\n    return hashlib.sha256(payload.encode(\"utf-8\")).hexdigest()\n\n\ndef canonical_answer(value):\n    value = \" \".join(value.strip().strip('\"').casefold().split())\n    match = re.fullmatch(\n        r\"(\\$)?\\s*([+-]?\\d[\\d,]*(?:\\.\\d+)?)\\s*(thousand|million|billion|trillion)?\\s*(%)?\",\n        value,\n    )\n    if match:\n        currency, number, scale, percent = match.groups()\n        multiplier = {None: 1, \"thousand\": 10**3, \"million\": 10**6,\n                      \"billion\": 10**9, \"trillion\": 10**12}[scale]\n        number = Decimal(number.replace(\",\", \"\")) * multiplier\n        return (currency or \"\") + format(number.normalize(), \"f\") + (percent or \"\")\n    return value\n\n\ndef score_answer(response, answers):\n    return float(canonical_answer(response) in {canonical_answer(a) for a in answers})\n\n\ndef load_runs(run_dirs):\n    runs, excluded = [], set()\n    for label, directory in run_dirs.items():\n        root = Path(directory)\n        adapter = root / \"adapter\"\n        required = [root / \"run_config.json\", root / \"split_manifest.json\",\n                    adapter / \"adapter_config.json\", adapter / \"adapter_model.safetensors\",\n                    adapter / \"tokenizer_config.json\"]\n        missing = [str(p) for p in required if not p.is_file()]\n        if missing:\n            raise ValueError(f\"Incomplete run {label}: {missing}\")\n        config = json.loads(required[0].read_text())\n        if not re.fullmatch(r\"[0-9a-f]{40}\", config.get(\"model_sha\", \"\")):\n            raise ValueError(f\"Missing immutable model revision: {label}\")\n        adapter_config = json.loads((adapter / \"adapter_config.json\").read_text())\n        if adapter_config.get(\"base_model_name_or_path\") != config[\"MODEL_ID\"]:\n            raise ValueError(f\"Adapter/base model mismatch: {label}\")\n        if config.get(\"MAX_DOC_CHARS\") != 32768:\n            raise ValueError(\"Overlap checking currently requires MAX_DOC_CHARS=32768 in every run.\")\n        manifest = json.loads(required[1].read_text())\n        if set(manifest) != {\"train\", \"eval\"} or not manifest[\"train\"]:\n            raise ValueError(f\"Invalid split manifest: {label}\")\n        for split in manifest.values():\n            for row in split:\n                digest = row.get(\"sha256\", \"\")\n                if not re.fullmatch(r\"[0-9a-f]{64}\", digest):\n                    raise ValueError(f\"Invalid document hash: {label}\")\n                excluded.add(digest)\n        runs.append(dict(label=label, model_id=config[\"MODEL_ID\"],\n                         model_sha=config[\"model_sha\"], dataset_id=config[\"DATASET_ID\"],\n                         dataset_sha=config[\"dataset_sha\"], adapter=str(adapter),\n                         training_config=config,\n                         manifest_sha256=hashlib.sha256(required[1].read_bytes()).hexdigest(),\n                         adapter_sha256=hashlib.sha256(\n                             (adapter / \"adapter_model.safetensors\").read_bytes()).hexdigest()))\n    if not runs:\n        raise ValueError(\"At least one completed run is required.\")\n    if len({(r[\"dataset_id\"], r[\"dataset_sha\"]) for r in runs}) != 1:\n        raise ValueError(\"Training runs use different dataset revisions; align their source data first.\")\n    return runs, excluded\n\n\ndef build_suite(rows, excluded, dataset_sha, document_count=200, question_count=20):\n    if not 1 <= question_count <= document_count:\n        raise ValueError(\"Require 1 <= question_count <= document_count.\")\n    documents, questions, seen = [], [], set()\n    for row in rows:\n        if not isinstance(row.get(\"source\"), str):\n            continue\n        full = normalize_text(row[\"source\"])\n        capped = full[:32768].strip()\n        digest = normalized_hash(capped)\n        if digest in excluded or digest in seen or len(capped) < 80:\n            continue\n        seen.add(digest)\n        passage = capped[:1200].strip()\n        match = MONEY.search(passage)\n        if not match and document_count - len(documents) <= question_count - len(questions):\n            continue\n        document_id = f\"financial_{len(documents):04d}\"\n        document = dict(id=document_id, text=passage, document_sha256=digest,\n                        capped_document=capped, passage_sha256=normalized_hash(passage),\n                        source_label=str(row.get(\"dataset\", \"unknown\")),\n                        source_file=str(row.get(\"source_file\", \"\")), source_index=str(row.get(\"index\", \"\")))\n        if len(documents) < document_count:\n            documents.append(document)\n            if match and len(questions) < question_count:\n                questions.append(dict(\n                    id=f\"amount_{len(questions):04d}\", document_id=document_id,\n                    context=passage,\n                    question=\"What is the first dollar-denominated amount in the passage? Copy the amount including $ and any scale word. Return only the amount.\",\n                    answers=[match.group(0)], category=\"literal_amount_extraction\",\n                ))\n        if len(documents) == document_count and len(questions) == question_count:\n            break\n    if len(documents) != document_count or len(questions) != question_count:\n        raise ValueError(\"Insufficient unseen passages with dollar amounts. Increase scan budget or choose a new suite.\")\n    suite = dict(schema_version=1, dataset_sha=dataset_sha, documents=documents,\n                 questions=questions, purpose=\"pilot\", reference_policy=\"literal_first_dollar_amount\")\n    audit_suite(suite, excluded)\n    return suite\n\n\ndef audit_suite(suite, excluded):\n    if suite.get(\"schema_version\") != 1 or not suite.get(\"documents\") or not suite.get(\"questions\"):\n        raise ValueError(\"Invalid or empty evaluation suite.\")\n    ids, hashes = set(), set()\n    documents = {}\n    for doc in suite[\"documents\"]:\n        if doc[\"id\"] in ids or doc[\"document_sha256\"] in hashes:\n            raise ValueError(\"Duplicate evaluation document.\")\n        ids.add(doc[\"id\"])\n        hashes.add(doc[\"document_sha256\"])\n        if normalized_hash(doc[\"capped_document\"]) != doc[\"document_sha256\"]:\n            raise ValueError(\"Document content/hash mismatch.\")\n        if doc[\"document_sha256\"] in excluded:\n            raise ValueError(\"Test document overlaps a recorded training/validation document.\")\n        if not doc[\"text\"] or doc[\"text\"] not in doc[\"capped_document\"]:\n            raise ValueError(\"Passage is not grounded in its source document.\")\n        if normalized_hash(doc[\"text\"]) != doc[\"passage_sha256\"]:\n            raise ValueError(\"Passage content/hash mismatch.\")\n        documents[doc[\"id\"]] = doc\n    question_ids = set()\n    for question in suite[\"questions\"]:\n        doc = documents.get(question[\"document_id\"])\n        if question[\"id\"] in question_ids or doc is None or question[\"context\"] != doc[\"text\"]:\n            raise ValueError(\"Invalid question ID, source, or context.\")\n        question_ids.add(question[\"id\"])\n        if not question[\"answers\"] or not all(isinstance(a, str) and a.strip() for a in question[\"answers\"]):\n            raise ValueError(\"Reference answers are required.\")\n        if question.get(\"category\") == \"literal_amount_extraction\":\n            match = MONEY.search(question[\"context\"])\n            if not match or question[\"answers\"] != [match.group(0)]:\n                raise ValueError(\"Extraction reference does not match the passage.\")\n        elif not question.get(\"reviewed\", False):\n            raise ValueError(\"Custom calculation/comprehension questions require reviewed=true.\")\n\n\ndef qa_prompt(doc):\n    return f\"Passage:\\n{doc['context']}\\n\\nQuestion: {doc['question']}\\nAnswer:\"\n\n\ndef qa_results(doc, results):\n    return {\"acc\": score_answer(results[0], doc[\"answers\"])}\n\n\ndef rolling_results(doc, results):\n    return {\"bits_per_byte\": (float(results[0]), len(doc[\"text\"].encode(\"utf-8\")))}\n\n\ndef aggregate_bpb(items):\n    return -sum(ll for ll, _ in items) / (sum(count for _, count in items) * math.log(2))\n\n\ndef make_tasks(suite_path):\n    from lm_eval.api.task import ConfigurableTask\n\n    suite = json.loads(Path(suite_path).read_text())\n    stem = Path(suite_path).parent\n    text_file, qa_file = stem / \"financial_text.jsonl\", stem / \"financial_qa.jsonl\"\n    for path, records in [(text_file, suite[\"documents\"]), (qa_file, suite[\"questions\"])]:\n        path.write_text(\"\".join(json.dumps(r, ensure_ascii=False) + \"\\n\" for r in records))\n    common = dict(dataset_path=\"json\", test_split=\"test\", num_fewshot=0)\n    text = ConfigurableTask(config=dict(\n        **common, task=\"financial_prose\", dataset_kwargs={\"data_files\": {\"test\": str(text_file)}},\n        output_type=\"loglikelihood_rolling\", doc_to_text=\"\", doc_to_target=\"{{text}}\",\n        process_results=rolling_results,\n        metric_list=[dict(metric=\"bits_per_byte\", aggregation=aggregate_bpb, higher_is_better=False)],\n    ))\n    qa = ConfigurableTask(config=dict(\n        **common, task=\"financial_qa\", dataset_kwargs={\"data_files\": {\"test\": str(qa_file)}},\n        output_type=\"generate_until\", doc_to_text=qa_prompt, doc_to_target=lambda d: d[\"answers\"][0],\n        process_results=qa_results,\n        generation_kwargs=dict(do_sample=False, max_gen_toks=48, until=[\"\\n\"]),\n        metric_list=[dict(metric=\"acc\", aggregation=\"mean\", higher_is_better=True)],\n    ))\n    return [text, qa]\n\n\ndef evaluate_runs(runs, suite_path, output_dir, excluded, max_length=1024):\n    import torch\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    from peft import PeftModel\n    from lm_eval import simple_evaluate\n    from lm_eval.models.huggingface import HFLM\n\n    if not torch.cuda.is_available():\n        raise RuntimeError(\"Select a Colab NVIDIA GPU runtime.\")\n    suite = json.loads(Path(suite_path).read_text())\n    audit_suite(suite, excluded)\n    digest = suite_digest(suite)\n    output = Path(output_dir)\n    output.mkdir(parents=True, exist_ok=False)\n    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16\n    freeze = dict(suite_sha256=digest, dataset_sha=suite[\"dataset_sha\"],\n                  purpose=suite[\"purpose\"], runs=runs, max_length=max_length,\n                  dtype=str(dtype), quantization=\"none\", gpu=torch.cuda.get_device_name(0),\n                  note=\"Qwen QLoRA adapter evaluated on an unquantized base; differs from its notebook's NF4 baseline.\")\n    (output / \"evaluation_config.json\").write_text(json.dumps(freeze, indent=2))\n    summaries = []\n    for run in runs:\n        if run[\"dataset_sha\"] != suite[\"dataset_sha\"]:\n            raise ValueError(\"Evaluation/training dataset revision mismatch.\")\n        tokenizer = AutoTokenizer.from_pretrained(run[\"adapter\"])\n        if tokenizer.pad_token_id is None:\n            tokenizer.pad_token = tokenizer.eos_token\n        tokenizer.padding_side = \"left\"\n        for question in suite[\"questions\"]:\n            length = len(tokenizer(qa_prompt(question), add_special_tokens=False)[\"input_ids\"])\n            if length + 48 > max_length:\n                raise ValueError(f\"Question {question['id']} exceeds the shared context budget for {run['label']}.\")\n        for mode in (\"baseline\", \"adapted\"):\n            tasks = make_tasks(suite_path)\n            model = wrapper = None\n            try:\n                model = AutoModelForCausalLM.from_pretrained(\n                    run[\"model_id\"], revision=run[\"model_sha\"], torch_dtype=dtype,\n                    device_map={\"\": 0}, attn_implementation=\"sdpa\",\n                )\n                if max_length > model.config.max_position_embeddings:\n                    raise ValueError(\"Evaluation context exceeds model capacity.\")\n                if mode == \"adapted\":\n                    model = PeftModel.from_pretrained(model, run[\"adapter\"], is_trainable=False)\n                model.eval()\n                model.config.use_cache = True\n                wrapper = HFLM(pretrained=model, tokenizer=tokenizer, batch_size=1,\n                               max_length=max_length, add_bos_token=False)\n                torch.cuda.synchronize()\n                torch.cuda.reset_peak_memory_stats()\n                start = time.perf_counter()\n                result = simple_evaluate(\n                    model=wrapper, tasks=tasks, num_fewshot=0, log_samples=True,\n                    apply_chat_template=False, bootstrap_iters=1000,\n                    random_seed=42, numpy_random_seed=42, torch_random_seed=42,\n                    fewshot_random_seed=42,\n                )\n                torch.cuda.synchronize()\n                seconds = time.perf_counter() - start\n                metrics = result[\"results\"]\n                row = dict(model=run[\"label\"], model_id=run[\"model_id\"], mode=mode,\n                           suite_sha256=digest,\n                           bits_per_byte=metrics[\"financial_prose\"][\"bits_per_byte,none\"],\n                           qa_accuracy=metrics[\"financial_qa\"][\"acc,none\"],\n                           seconds=seconds, peak_allocated_gib=torch.cuda.max_memory_allocated() / 2**30,\n                           model_parameters=sum(p.numel() for p in model.parameters()),\n                           document_count=len(suite[\"documents\"]), question_count=len(suite[\"questions\"]),\n                           dtype=str(dtype))\n                if not all(math.isfinite(row[k]) for k in (\"bits_per_byte\", \"qa_accuracy\")):\n                    raise ValueError(\"Non-finite evaluation scores.\")\n                summaries.append(row)\n                run_output = output / f\"{run['label']}_{mode}\"\n                run_output.mkdir()\n                (run_output / \"harness_results.json\").write_text(json.dumps(\n                    result, indent=2, ensure_ascii=False, default=str))\n                with (run_output / \"samples.jsonl\").open(\"w\") as handle:\n                    for task_name, samples in result.get(\"samples\", {}).items():\n                        for sample in samples:\n                            handle.write(json.dumps(dict(task=task_name, **sample), ensure_ascii=False, default=str) + \"\\n\")\n                with (output / \"comparison.csv\").open(\"w\", newline=\"\") as handle:\n                    writer = csv.DictWriter(handle, fieldnames=list(row))\n                    writer.writeheader()\n                    writer.writerows(summaries)\n                (output / \"comparison.json\").write_text(json.dumps(summaries, indent=2, allow_nan=False))\n                print(json.dumps(row, indent=2))\n            finally:\n                del wrapper, model, tasks\n                gc.collect()\n                torch.cuda.empty_cache()\n    paired = []\n    for run in runs:\n        baseline, adapted = [r for r in summaries if r[\"model\"] == run[\"label\"]]\n        paired.append(dict(model=run[\"label\"],\n                           qa_accuracy_change=adapted[\"qa_accuracy\"] - baseline[\"qa_accuracy\"],\n                           bits_per_byte_change=adapted[\"bits_per_byte\"] - baseline[\"bits_per_byte\"]))\n    (output / \"improvement.json\").write_text(json.dumps(paired, indent=2, allow_nan=False))\n    return summaries\n")


## 3. Mount Drive and select the completed training runs

By default, select the latest **complete** run in each notebook's output folder.
Review the printed paths before continuing. To select particular runs, replace
RUN_DIRS=None with a dictionary mapping qwen/smollm2/pythia to their run folders,
not their adapter subfolders.

Every run must include a saved adapter, tokenizer, run_config.json, and
split_manifest.json. The audit rejects incomplete runs, base-model mismatches,
and different dataset revisions. Do not change selection after freezing a suite.


In [ ]:
from google.colab import drive
from importlib.metadata import version
from datetime import datetime, timezone
import itertools
from datasets import load_dataset
import torch

drive.mount("/content/drive")
RUN_DIRS = None
ROOTS = {
    "qwen": "/content/drive/MyDrive/qwen_financial_qlora",
    "smollm2": "/content/drive/MyDrive/smollm2_360m_financial_lora",
    "pythia": "/content/drive/MyDrive/pythia_410m_financial_lora",
}
if RUN_DIRS is None:
    RUN_DIRS = {}
    for label, folder in ROOTS.items():
        for candidate in sorted(Path(folder).glob("run_*"), reverse=True):
            try:
                load_runs({label: candidate})
            except (ValueError, KeyError, OSError):
                continue
            RUN_DIRS[label] = str(candidate)
            break
        if label not in RUN_DIRS:
            raise ValueError(f"No complete run found for {label} in {folder}. Finish its adapter save cell or set RUN_DIRS explicitly.")
if set(RUN_DIRS) != set(ROOTS):
    raise ValueError("Supply exactly qwen, smollm2, and pythia run folders.")
runs, excluded = load_runs(RUN_DIRS)
if version("lm_eval") != "0.4.9" or version("transformers") != "4.51.3":
    raise RuntimeError("Restart after installing the pinned evaluation stack.")
for run in runs:
    print(run["label"], run["model_id"], run["model_sha"], run["adapter"])
print("Excluded exact document hashes:", len(excluded))


## 4. Create or verify a frozen pilot suite

The first run streams additional source documents from the exact training dataset
revision, excluding every recorded training **and validation** document. It stores
the capped source document and a 1,200-character passage, plus literal extraction
references. The question asks for the **first dollar amount**, not an inferred
financial answer. Numerical scoring preserves currency, scale, and percent units.

The default is a 20-question pilot. Review its contents before model evaluation.
To run 200 passages / 100 questions, change both counts and SUITE_NAME.
Existing frozen suites cannot be silently regenerated or edited; their stored
SHA256 is verified. Use a new suite name for any changes.

This writes selected corpus text to your own Drive. Check source terms before
redistributing. Exclusion covers exact documents under the notebooks' normalization
and 32,768-character cap; it does not prove absence of near duplicates.


In [ ]:
EVAL_ROOT = Path("/content/drive/MyDrive/llm_flow_evaluation")
SUITE_NAME = "financial_pilot_v1"
DOCUMENT_COUNT = 20
QUESTION_COUNT = 20
SCAN_ROWS = 100000
suite_dir = EVAL_ROOT / "suites" / SUITE_NAME
suite_path = suite_dir / "suite.json"
fingerprint_path = suite_dir / "freeze.json"
if suite_path.is_file():
    suite = json.loads(suite_path.read_text())
    audit_suite(suite, excluded)
    frozen = json.loads(fingerprint_path.read_text())
    if suite_digest(suite) != frozen["suite_sha256"]:
        raise ValueError("Frozen suite changed. Restore it or create a new suite name.")
    if len(suite["documents"]) != DOCUMENT_COUNT or len(suite["questions"]) != QUESTION_COUNT:
        raise ValueError("Counts differ from the frozen suite. Choose a new SUITE_NAME.")
else:
    from collections import Counter
    dataset = load_dataset(runs[0]["dataset_id"], split="train",
                          revision=runs[0]["dataset_sha"], streaming=True)
    stream = dataset.shuffle(seed=1729, buffer_size=10000)
    suite = build_suite(itertools.islice(stream, SCAN_ROWS), excluded,
                        runs[0]["dataset_sha"], DOCUMENT_COUNT, QUESTION_COUNT)
    suite_dir.mkdir(parents=True, exist_ok=False)
    suite_path.write_text(json.dumps(suite, indent=2, ensure_ascii=False))
    fingerprint_path.write_text(json.dumps(
        {"suite_sha256": suite_digest(suite),
         "dataset_sha": suite["dataset_sha"],
         "excluded_hashes_sha256": hashlib.sha256("\n".join(sorted(excluded)).encode()).hexdigest()},
        indent=2))
if suite["dataset_sha"] != runs[0]["dataset_sha"]:
    raise ValueError("Frozen suite belongs to a different dataset revision.")
print("Frozen suite:", suite_path)
print("SHA256:", suite_digest(suite))
print("Documents:", len(suite["documents"]), "Questions:", len(suite["questions"]))
for item in suite["questions"][:3]:
    print("\n", item["context"], "\nQuestion:", item["question"], "\nReference:", item["answers"])


## 5. Run six evaluations

Runs original then adapted models for each family, freeing GPU memory between
loads. All use the same 1,024-token maximum context, greedy generation, no chat
template, and batch size 1. The notebook checks that QA prompts fit, so no model
silently loses passage context.

All bases are evaluated **without quantization** in BF16 when natively supported,
otherwise FP16. This makes baseline/adapted precision match. For the QLoRA-trained
Qwen adapter, this is deliberately a different evaluation precision from the
training notebook's NF4 baseline; do not mix those old metrics into this table.

bits_per_byte: lower is better, weighted by UTF-8 text bytes.
qa_accuracy: higher is better, requiring a whole-response reference match with
numeric scale normalization (for example $1 million equals $1,000,000).
Execution time is combined scoring time, not a pure inference throughput benchmark.


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("Select an NVIDIA GPU runtime.")
MAX_CONTEXT = 1024
timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
output_dir = EVAL_ROOT / "results" / f"{SUITE_NAME}_{timestamp}"
summary = evaluate_runs(runs, suite_path, output_dir, excluded, max_length=MAX_CONTEXT)
(output_dir / "evaluation_utils.py").write_text(
    Path("/content/evaluation_utils_snapshot.py").read_text()
)
(output_dir / "requirements_freeze.txt").write_text(
    subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
)
print("Saved comparison and predictions:", output_dir)


## 6. View and export the comparison

Each model/mode folder contains full harness results and samples.jsonl with
per-example documents, responses, and scores. The results directory contains
comparison.csv, comparison.json, improvement.json, evaluation configuration, a
package snapshot, and evaluation source. Scores are saved after each completed
model/mode, so a later failure does not discard earlier results.

Do not repeatedly tune adapters against this pilot and then call it an untouched
test. Freeze an independent final suite for the final report. Custom reviewed
extraction/calculation questions can be supplied in a new suite using the same
question schema; set category to your task type and reviewed=true. Verify source
context and answers before freezing. The included pilot alone does not support
claims about broad reasoning or factual accuracy.


In [ ]:
import pandas as pd
table = pd.DataFrame(summary)
display(table[["model", "mode", "bits_per_byte", "qa_accuracy", "seconds", "peak_allocated_gib"]])
display(pd.read_json(output_dir / "improvement.json"))


## Sources and validation

- [Harness v0.4.9](https://github.com/EleutherAI/lm-evaluation-harness/tree/v0.4.9)
- [Custom task interface](https://github.com/EleutherAI/lm-evaluation-harness/blob/v0.4.9/lm_eval/api/task.py)
- [Hugging Face model backend](https://github.com/EleutherAI/lm-evaluation-harness/blob/v0.4.9/lm_eval/models/huggingface.py)

No model scores are prepopulated. Local tests cover hashing, contamination rejection,
reference validation, packing-independent question scoring, and incomplete run
detection. A complete Colab install/download/GPU evaluation is still required.
